# Harmonic radius: rational tuning presets

This notebook loads a ratio-only Hexatone preset, converts every scale degree to an exact prime-exponent monzo, and explores degree and collection radii. Scale indices shown here are zero-based and preserve the order in the source JSON.

In [3]:
cwd = Path.cwd()
print(cwd)
print(cwd.name)

/Users/marcsabat/Dev/hexatone/_harmonic-analysis/notebooks
notebooks


In [4]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
preset_relative_path = Path("data") / "sabat-the-tree-modulations.json"
candidates = [cwd, cwd / "_harmonic-analysis", *cwd.parents]
project_dir = next(
    (path for path in candidates if (path / preset_relative_path).is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError(
        f"Can't locate the analysis project from {cwd}. Open the notebook from the repo or _harmonic-analysis folder."
    )
sys.path.insert(0, str(project_dir / "src"))

from tuning_preset_json import load_tuning_preset
from radius import (
    collection_radius, combination_count, degree_radius,
    iter_collection_radii,
)

preset_path = project_dir / preset_relative_path
preset = load_tuning_preset(preset_path)
print(preset.name, "—", preset.degree_count, "scale entries")

Sabat: The Tree + Modulations — 81 scale entries


## Degree radius from the assigned zero

For an individual interval, the reduced numerator and denominator are treated as its two partials. The full radius includes powers of 2; the odd radius removes them. The preset's `reference_degree` and `fundamental` remain metadata and do not silently redefine the rational zero. Supply a `zero_monzo` explicitly to `degree_radius` when you want another assigned zero.

In [6]:
degree_rows = []
for index, (ratio, monzo) in enumerate(zip(preset.ratios, preset.monzos)):
    radius = degree_radius(monzo, preset.zero_monzo)
    degree_rows.append({
        "degree": index + 1, "ratio": ratio, "monzo": monzo,
        "HR": radius.harmonic_radius, "odd HR": radius.odd_radius,
        "log2 HR": radius.log2_harmonic_radius,
        "prime-limit HR": radius.prime_limit_radius,
    })
degree_rows[:12]

[{'degree': 1,
  'ratio': '129/128',
  'monzo': {2: -7, 3: 1, 43: 1},
  'HR': 128.49902723367208,
  'odd HR': 11.357816691600547,
  'log2 HR': 7.005613627711627,
  'prime-limit HR': 34.520297733469626},
 {'degree': 2,
  'ratio': '65/64',
  'monzo': {2: -6, 5: 1, 13: 1},
  'HR': 64.4980619863884,
  'odd HR': 8.06225774829855,
  'log2 HR': 6.011183906514227,
  'prime-limit HR': 18.134962824817467},
 {'degree': 3,
  'ratio': '33/32',
  'monzo': {2: -5, 3: 1, 11: 1},
  'HR': 32.49615361854385,
  'odd HR': 5.744562646538029,
  'log2 HR': 5.022197059679227,
  'prime-limit HR': 12.34586858289287},
 {'degree': 4,
  'ratio': '133/128',
  'monzo': {2: -7, 7: 1, 19: 1},
  'HR': 130.47605144240077,
  'odd HR': 11.532562594670797,
  'log2 HR': 7.027641217750595,
  'prime-limit HR': 28.360331425490003},
 {'degree': 5,
  'ratio': '135/128',
  'monzo': {2: -7, 3: 3, 5: 1},
  'HR': 131.45341380123992,
  'odd HR': 11.61895003862225,
  'log2 HR': 7.038407798525416,
  'prime-limit HR': 20.38853093816547},

## Collection radius and hand-check

Collection radius translates all selected pitches together to their nearest common fundamental. This is not the mean of the individual degree radii. For the familiar dyad 3/2 and 5/4, the common-fundamental partials are 6 and 5.

In [7]:
from monzo import ratio_to_monzo

checked = collection_radius((ratio_to_monzo("3/2"), ratio_to_monzo("5/4")))
{
    "partials": checked.partials,
    "harmonic radius": checked.harmonic_radius,
    "odd radius": checked.odd_radius,
    "prime-limit radius": checked.prime_limit_radius,
    "odd prime-limit radius": checked.odd_prime_limit_radius,
}

{'partials': (6, 5),
 'harmonic radius': 5.477225575051659,
 'odd radius': 3.8729833462074166,
 'prime-limit radius': 4.605779351596906,
 'odd prime-limit radius': 3.8729833462074166}

## Degree selections and subsets

Set `selected_degrees` to zero-based indices, or leave it as `None` to analyse the full scale. Select subset sizes explicitly: an 81-degree scale has 3,240 dyads, 85,320 triads, and 1,663,740 tetrads. Results are generated lazily, so inspect the count before materialising a large result table.

In [ ]:
selected_degrees = None  # e.g. [10, 20, 42, 60]
selected_count = preset.degree_count if selected_degrees is None else len(selected_degrees)
{size: combination_count(selected_count, size) for size in (2, 3, 4)}

In [ ]:
subset_size = 3
results = iter_collection_radii(
    preset.monzos, subset_size, degree_indices=selected_degrees
)
# Preview only: don't collect every combination into memory.
next(results)